# Phase 3 — Custom Convolutional Neural Network (Scratch CNN)

**Project:** RealWaste Waste Classification | **Course:** Introduction to Machine Learning (IML)

In Phase 2, we built classical baselines using hand-crafted features (HOG + Color Histograms). In Phase 3, we build and train a **custom 4-stage Deep Convolutional Neural Network entirely from scratch** (no pretrained weights / no transfer learning).

### Ablation Study (Regularization & Generalization):
To evaluate how regularization mitigates overfitting on real-world landfill data (~3,326 training images), we train and contrast two variants:
1. **Model A — Scratch CNN Baseline:** No data augmentation, Dropout = 0.0 (demonstrates overfitting / high variance).
2. **Model B — Regularized Scratch CNN:** Data augmentation (flips, rotations, jitter, cropping) + Dropout = 0.4 + Weight Decay + Learning Rate Scheduler.

>  **Recommended runtime:** GPU (T4 in Colab). Training takes ~1-2 minutes per model on GPU.

## 3.0 — Runtime & Environment Setup

In [ ]:
import sys
import os
import time
import copy
import json
from pathlib import Path

IS_COLAB  = 'google.colab' in sys.modules
IS_KAGGLE = 'kaggle_secrets' in sys.modules
IS_LOCAL  = not IS_COLAB and not IS_KAGGLE

print(f'Environment: {"Google Colab" if IS_COLAB else ("Kaggle" if IS_KAGGLE else "Local")}')

In [ ]:
# Install dependencies if needed
%pip install -q torch torchvision scikit-learn matplotlib seaborn pandas numpy pillow tqdm

In [ ]:
import torch

# Check device (GPU is strongly recommended)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')
if device.type == 'cuda':
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'Allocated memory: {torch.cuda.memory_allocated(0)/(1024**2):.1f} MB')

In [ ]:
# Mount Drive & configure directories
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/IML_RealWaste')
else:
    PROJECT_ROOT = Path('.').resolve().parent

DATA_DIR    = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
METRICS_DIR = RESULTS_DIR  / 'metrics'
PLOTS_DIR   = RESULTS_DIR  / 'plots'
MODELS_DIR  = RESULTS_DIR  / 'models'

for d in [METRICS_DIR, PLOTS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'Results dir  : {RESULTS_DIR}')

## 3.1 — Load Shared Dataset Split & Resolve Image Paths

In [ ]:
import pandas as pd

# Load class metadata
with open(DATA_DIR / 'dataset_meta.json') as f:
    meta = json.load(f)

CLASSES   = meta['classes']
LABEL2IDX = meta['label2idx']
IDX2LABEL = {int(v): k for k, v in LABEL2IDX.items()}
N_CLASSES = len(CLASSES)

df_train = pd.read_csv(DATA_DIR / 'train.csv')
df_val   = pd.read_csv(DATA_DIR / 'val.csv')
df_test  = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train samples: {len(df_train):>5}')
print(f'Val samples  : {len(df_val):>5}')
print(f'Test samples : {len(df_test):>5}')
print(f'Classes ({N_CLASSES}): {CLASSES}')

In [ ]:
# Find dataset root on disk and create a dynamic path resolver
def locate_dataset_root(search_base: Path) -> Path:
    for root, dirs, files in os.walk(search_base):
        if set(CLASSES).issubset(set(dirs)):
            return Path(root)
    return search_base

DATASET_ROOT = locate_dataset_root(DATA_DIR)
if not DATASET_ROOT.exists():
    DATASET_ROOT = locate_dataset_root(PROJECT_ROOT)
print(f'Dataset root identified at: {DATASET_ROOT}')

def resolve_path(filepath_str: str) -> Path:
    """Resolves image path whether running in Colab or local environment."""
    p = Path(filepath_str)
    if p.exists():
        return p
    # Fallback to dataset root using class folder and filename
    parts = p.parts
    if len(parts) >= 2:
        cls_name, fname = parts[-2], parts[-1]
        candidate = DATASET_ROOT / cls_name / fname
        if candidate.exists():
            return candidate
    return p

# Verify first image
sample_check = resolve_path(df_train['filepath'].iloc[0])
print(f'Sample verification: {sample_check} -> Exists: {sample_check.exists()}')

## 3.2 — PyTorch Dataset & Data Augmentation Pipelines

- **Evaluation Pipeline (Val & Test):** Deterministic resize (224×224) and normalization with standard ImageNet statistics.
- **Baseline Training (No Augmentation):** Identical deterministic preprocessing without data transformations.
- **Regularized Training (With Augmentation):** Live random transformations to artificially simulate viewing angles, orientations, and lighting in landfill environments.

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMAGE_SIZE = (224, 224)
NORM_MEAN  = [0.485, 0.456, 0.406]
NORM_STD   = [0.229, 0.224, 0.225]

# 1. Deterministic transforms (No augmentation)
transform_eval = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

# 2. Augmentation transforms for Regularized model
transform_augmented = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

class RealWasteDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = resolve_path(row['filepath'])
        image = Image.open(img_path).convert('RGB')
        label = int(row['label_idx'])

        if self.transform:
            image = self.transform(image)

        return image, label

# Create datasets
ds_train_baseline   = RealWasteDataset(df_train, transform=transform_eval)
ds_train_augmented  = RealWasteDataset(df_train, transform=transform_augmented)
ds_val              = RealWasteDataset(df_val,   transform=transform_eval)
ds_test             = RealWasteDataset(df_test,  transform=transform_eval)

BATCH_SIZE = 32
NUM_WORKERS = 2 if IS_COLAB else 0

loader_val  = DataLoader(ds_val,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
loader_test = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f'Validation batches: {len(loader_val)}')
print(f'Test batches      : {len(loader_test)}')

## 3.3 — Define Custom 4-Stage CNN Architecture

Our network uses 4 progressive convolution stages ($3 \to 32 \to 64 \to 128 \to 256$), each equipped with:
- $3\times 3$ 2D Convolution
- Batch Normalization (stabilizes deep feature representations and gradient propagation)
- ReLU non-linearity
- $2\times 2$ Max-Pooling (halves spatial resolution)
- Adaptive Global Average Pooling ($1\times 1$) to eliminate parameter explosions
- Dropout layer with configurable probability $p$

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class RealWasteScratchCNN(nn.Module):
    def __init__(self, num_classes=9, dropout_rate=0.4):
        super().__init__()
        # Stage 1: (3, 224, 224) -> (32, 112, 112)
        self.stage1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # Stage 2: (32, 112, 112) -> (64, 56, 56)
        self.stage2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # Stage 3: (64, 56, 56) -> (128, 28, 28)
        self.stage3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # Stage 4: (128, 28, 28) -> (256, 14, 14)
        self.stage4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # Global pooling and classifier head
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.classifier(x)
        return x

# Parameter inspection
sample_model = RealWasteScratchCNN(num_classes=N_CLASSES, dropout_rate=0.4)
total_params = sum(p.numel() for p in sample_model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params:,} (~{total_params/1e6:.2f}M)')

## 3.4 — Training Engine with Checkpointing & Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds, all_targets = [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc  = accuracy_score(all_targets, all_preds)
    return epoch_loss, epoch_acc


def evaluate_model(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(labels.cpu().numpy())

    val_loss = running_loss / len(loader.dataset)
    val_acc  = accuracy_score(all_targets, all_preds)
    val_f1   = f1_score(all_targets, all_preds, average='macro')
    return val_loss, val_acc, val_f1, all_preds, all_targets


def train_experiment(model, train_loader, val_loader, criterion, optimizer,
                     scheduler=None, num_epochs=20, exp_name='Model', device=device):
    print(f'=== Starting Training: {exp_name} ({num_epochs} Epochs) ===')
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
    best_f1 = -1.0
    best_model_weights = None
    start_time = time.time()

    for epoch in range(1, num_epochs + 1):
        t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        v_loss, v_acc, v_f1, _, _ = evaluate_model(model, val_loader, criterion, device)

        if scheduler:
            scheduler.step(v_loss)

        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)
        history['val_f1'].append(v_f1)

        # Checkpoint best model
        is_best = v_f1 > best_f1
        if is_best:
            best_f1 = v_f1
            best_model_weights = copy.deepcopy(model.state_dict())

        star = 'Best' if is_best else ''
        print(f'Epoch {epoch:>2}/{num_epochs} | Train Loss: {t_loss:.4f} Acc: {t_acc*100:.1f}% | Val Loss: {v_loss:.4f} Acc: {v_acc*100:.1f}% F1: {v_f1:.4f}{star}')

    total_time = time.time() - start_time
    print(f'{exp_name} finished in {total_time:.1f}s | Best Val Macro F1: {best_f1:.4f}')
    model.load_state_dict(best_model_weights)
    return model, history, total_time

## 3.5 — Experiment 1: Scratch CNN Baseline (No Augmentation, Dropout = 0.0)

This experiment serves as the unregularized reference baseline to measure the severity of overfitting when training deep networks from scratch on limited samples.

In [ ]:
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

loader_train_baseline = DataLoader(ds_train_baseline, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

model_baseline = RealWasteScratchCNN(num_classes=N_CLASSES, dropout_rate=0.0).to(device)
criterion = nn.CrossEntropyLoss()
optimizer_baseline = torch.optim.Adam(model_baseline.parameters(), lr=1e-3, weight_decay=0.0)

model_baseline, history_baseline, time_baseline = train_experiment(
    model_baseline,
    train_loader=loader_train_baseline,
    val_loader=loader_val,
    criterion=criterion,
    optimizer=optimizer_baseline,
    num_epochs=20,
    exp_name='Scratch CNN (Baseline / No Aug)',
    device=device
)

# Save model checkpoint
torch.save(model_baseline.state_dict(), MODELS_DIR / 'best_scratch_cnn_baseline.pth')
print(f'Saved baseline weights to: {MODELS_DIR}/best_scratch_cnn_baseline.pth')

## 3.6 — Experiment 2: Regularized Scratch CNN (Augmentation + Dropout 0.4 + Weight Decay + Scheduler)

This experiment tests how live geometric/color augmentations, weight decay, and dropout regularize the model and boost generalization on unseen waste images.

In [ ]:
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

loader_train_augmented = DataLoader(ds_train_augmented, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

model_regularized = RealWasteScratchCNN(num_classes=N_CLASSES, dropout_rate=0.4).to(device)
optimizer_reg = torch.optim.Adam(model_regularized.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_reg = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_reg, mode='min', factor=0.5, patience=3)

model_regularized, history_regularized, time_regularized = train_experiment(
    model_regularized,
    train_loader=loader_train_augmented,
    val_loader=loader_val,
    criterion=criterion,
    optimizer=optimizer_reg,
    scheduler=scheduler_reg,
    num_epochs=25,
    exp_name='Scratch CNN (Regularized + Augmented)',
    device=device
)

# Save model checkpoint
torch.save(model_regularized.state_dict(), MODELS_DIR / 'best_scratch_cnn_regularized.pth')
print(f'Saved regularized weights to: {MODELS_DIR}/best_scratch_cnn_regularized.pth')

## 3.7 — Evaluation on Unseen Test Set (713 Images)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. Test Baseline Model
loss_base, acc_base, f1_base, preds_base, targets_base = evaluate_model(model_baseline, loader_test, criterion, device)
report_base = classification_report(targets_base, preds_base, target_names=CLASSES, output_dict=True)
cm_base = confusion_matrix(targets_base, preds_base)

# 2. Test Regularized Model
loss_reg, acc_reg, f1_reg, preds_reg, targets_reg = evaluate_model(model_regularized, loader_test, criterion, device)
report_reg = classification_report(targets_reg, preds_reg, target_names=CLASSES, output_dict=True)
cm_reg = confusion_matrix(targets_reg, preds_reg)

# Comparison DataFrame
summary_cnn = pd.DataFrame([
    {
        'Model': 'Scratch CNN (Baseline / No Aug)',
        'Accuracy (%)': round(acc_base * 100, 2),
        'Macro F1': round(f1_base, 4),
        'Test Loss': round(loss_base, 4),
        'Train Time (s)': round(time_baseline, 1)
    },
    {
        'Model': 'Scratch CNN (Regularized + Aug)',
        'Accuracy (%)': round(acc_reg * 100, 2),
        'Macro F1': round(f1_reg, 4),
        'Test Loss': round(loss_reg, 4),
        'Train Time (s)': round(time_regularized, 1)
    }
])

print('=' * 65)
print('       SCRATCH CNN TEST RESULTS (Test Set: 713 Images)')
print('=' * 65)
print(summary_cnn.to_string(index=False))
print('=' * 65)

In [ ]:
# Per-class F1 comparison
df_perclass_cnn = pd.DataFrame({
    'Scratch CNN (Baseline)':    {cls: round(report_base[cls]['f1-score'], 3) for cls in CLASSES},
    'Scratch CNN (Regularized)': {cls: round(report_reg[cls]['f1-score'], 3)  for cls in CLASSES}
})
df_perclass_cnn['Gain (F1)'] = (df_perclass_cnn['Scratch CNN (Regularized)'] - df_perclass_cnn['Scratch CNN (Baseline)']).round(3)

print('\nPer-Class F1 Breakdown:')
print(df_perclass_cnn.to_string())

## 3.8 — Visualizations (Learning Curves, Confusion Matrices, and Multi-Tier Comparison)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── Plot 1: Learning Curves (Overfitting vs Regularization) ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(history_baseline['train_loss'], label='Baseline: Train Loss', color='crimson', linestyle='--')
axes[0].plot(history_baseline['val_loss'],   label='Baseline: Val Loss',   color='crimson', linewidth=2)
axes[0].plot(history_regularized['train_loss'], label='Regularized: Train Loss', color='forestgreen', linestyle='--')
axes[0].plot(history_regularized['val_loss'],   label='Regularized: Val Loss',   color='forestgreen', linewidth=2)
axes[0].set_title('Cross-Entropy Loss (Overfitting vs Regularization)', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot([a*100 for a in history_baseline['train_acc']], label='Baseline: Train Acc', color='crimson', linestyle='--')
axes[1].plot([a*100 for a in history_baseline['val_acc']],   label='Baseline: Val Acc',   color='crimson', linewidth=2)
axes[1].plot([a*100 for a in history_regularized['train_acc']], label='Regularized: Train Acc', color='forestgreen', linestyle='--')
axes[1].plot([a*100 for a in history_regularized['val_acc']],   label='Regularized: Val Acc',   color='forestgreen', linewidth=2)
axes[1].set_title('Accuracy (%) Over Epochs', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'scratch_cnn_learning_curves.png', dpi=150)
plt.show()
print(f'Saved: {PLOTS_DIR}/scratch_cnn_learning_curves.png')

In [ ]:
# ── Plot 2: Confusion Matrices ────────────────────────────────────────────────
SHORT = [c.replace(' ', '\n') for c in CLASSES]
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for idx, (title, cm, acc, f1) in enumerate([
    ('Scratch CNN Baseline', cm_base, acc_base, f1_base),
    ('Regularized Scratch CNN', cm_reg, acc_reg, f1_reg)
]):
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(
        cm_norm, annot=True, fmt='.2f',
        xticklabels=SHORT, yticklabels=SHORT,
        cmap='Greens' if idx == 1 else 'Reds', vmin=0, vmax=1,
        ax=axes[idx], cbar=False, annot_kws={'size': 8}
    )
    axes[idx].set_title(f'{title}\nAcc: {acc*100:.2f}% | Macro F1: {f1:.4f}', fontsize=12)
    axes[idx].set_xlabel('Predicted Label')
    axes[idx].set_ylabel('True Label')

plt.suptitle('Confusion Matrices on Test Set (Row-Normalized)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'scratch_cnn_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {PLOTS_DIR}/scratch_cnn_confusion_matrices.png')

In [ ]:
# ── Plot 3: Multi-Tier Comparison (Classical vs Scratch CNN) ──────────────────
# Load classical summary if available to build joint comparison
classical_summary_path = METRICS_DIR / 'classical_summary.csv'
if classical_summary_path.exists():
    df_class = pd.read_csv(classical_summary_path)
    combined_df = pd.concat([
        df_class[['Model', 'Accuracy (%)', 'Macro F1']],
        summary_cnn[['Model', 'Accuracy (%)', 'Macro F1']]
    ], ignore_index=True)

    fig, ax = plt.subplots(figsize=(11, 5))
    x = np.arange(len(combined_df))
    w = 0.35

    bars1 = ax.bar(x - w/2, combined_df['Accuracy (%)'], w, label='Accuracy (%)', color='steelblue')
    bars2 = ax.bar(x + w/2, combined_df['Macro F1'] * 100, w, label='Macro F1 (%)', color='darkorange')

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)

    ax.axhline(19.4, color='red', linestyle='--', linewidth=1, label='Majority baseline (19.4%)')
    ax.set_xticks(x)
    ax.set_xticklabels(combined_df['Model'], rotation=15, ha='right', fontsize=9)
    ax.set_ylabel('Score (%)')
    ax.set_ylim(0, 100)
    ax.set_title('Progression: Classical Baselines vs Scratch CNN Variants', fontsize=13)
    ax.legend()
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'scratch_vs_classical_comparison.png', dpi=150)
    plt.show()
    print(f'Saved: {PLOTS_DIR}/scratch_vs_classical_comparison.png')

## 3.9 — Save Metrics & Export Results

In [ ]:
# Save metric files
summary_cnn.to_csv(METRICS_DIR / 'scratch_cnn_summary.csv', index=False)
df_perclass_cnn.to_csv(METRICS_DIR / 'scratch_cnn_perclass_f1.csv')

pd.DataFrame(report_base).transpose().to_csv(METRICS_DIR / 'scratch_cnn_baseline_report.csv')
pd.DataFrame(report_reg).transpose().to_csv(METRICS_DIR / 'scratch_cnn_regularized_report.csv')

# Save training histories for plotting in final report
history_df = pd.DataFrame({
    'epoch': list(range(1, len(history_baseline['train_loss']) + 1)),
    'base_train_loss': history_baseline['train_loss'],
    'base_val_loss': history_baseline['val_loss'],
    'base_train_acc': history_baseline['train_acc'],
    'base_val_acc': history_baseline['val_acc'],
    'base_val_f1': history_baseline['val_f1'],
})
history_df.to_csv(METRICS_DIR / 'scratch_cnn_history.csv', index=False)

print(' Saved to results/metrics/:')
print('   scratch_cnn_summary.csv')
print('   scratch_cnn_perclass_f1.csv')
print('   scratch_cnn_baseline_report.csv')
print('   scratch_cnn_regularized_report.csv')
print('   scratch_cnn_history.csv')
print(' Phase 3 complete!')

In [ ]:
# Download all results as zip (run on Colab)
if IS_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive('/content/phase3_results', 'zip', str(RESULTS_DIR))
    files.download('/content/phase3_results.zip')
    print(' Phase 3 results zip download initiated!')